<a href="https://colab.research.google.com/github/madhugadi/LLM-fine-tuning/blob/main/LORA_QLORA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 95.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 17.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 64.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 29.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 59.5 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.0
    Uninstalling transformers-5.22.0:
      Successfully uninstalled transformers-5.22.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [trl]
ERROR: pip's dependency resolver does not currently take into ac

In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

In [3]:
# MODEL_NAME = "google/gemma-2-9b-it"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./TinyLlama"

In [4]:
dataset = load_dataset("Abirate/english_quotes")

README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl: reconstructing file:   0%|          |  0.00B /  647kB            

quotes.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags'],
        num_rows: 2508
    })
})

In [6]:
def format_example(example):
    quote = example["quote"].strip()
    tags = ", ".join(example["tags"]) if isinstance(example["tags"], list) else str(example["tags"])

    text = f"""<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
{quote}<end_of_turn>
<start_of_turn>model
{tags}<end_of_turn>"""
    return {"text": text}

In [7]:
train_dataset = dataset["train"].map(format_example)

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [8]:
train_dataset

Dataset({
    features: ['quote', 'author', 'tags', 'text'],
    num_rows: 2508
})

#### This format teaches the model how to respond to a user instruction by mapping input text to the expected output.

In [10]:
print(train_dataset['text'][0])

<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
“Be yourself; everyone else is already taken.”<end_of_turn>
<start_of_turn>model
be-yourself, gilbert-perreira, honesty, inspirational, misattributed-oscar-wilde, quote-investigator<end_of_turn>


In [11]:
print(train_dataset['text'][1])

<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
“I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.”<end_of_turn>
<start_of_turn>model
best, life, love, mistakes, out-of-control, truth, worst<end_of_turn>


In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [13]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [14]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [15]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [18]:
model = get_peft_model(base_model, lora_config)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [19]:
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [20]:
total_params = model.num_parameters()
trainable_params = model.num_parameters(only_trainable=True)

In [21]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape, param.dtype)

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.Size([16, 2048]) torch.float32
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.Size([2048, 16]) torch.float32
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight torch.Size([16, 2048]) torch.float32
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight torch.Size([256, 16]) torch.float32
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.Size([16, 2048]) torch.float32
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight torch.Size([256, 16]) torch.float32
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight torch.Size([16, 2048]) torch.float32
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight torch.Size([2048, 16]) torch.float32
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight torch.Size([16, 2048]) torch.float32
base_model.model.model.layers.

In [22]:
print(model.peft_config)

{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.2', base_model_name_or_path='TinyLlama/TinyLlama-1.1B-Chat-v1.0', revision=None, inference_mode=False, r=16, target_modules={'q_proj', 'o_proj', 'k_proj', 'v_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_we

In [23]:
config = model.peft_config["default"]

print("LoRA rank:", config.r)
print("LoRA alpha:", config.lora_alpha)
print("LoRA dropout:", config.lora_dropout)
print("Target modules:", config.target_modules)
print("Task type:", config.task_type)

LoRA rank: 16
LoRA alpha: 32
LoRA dropout: 0.05
Target modules: {'q_proj', 'o_proj', 'k_proj', 'v_proj'}
Task type: CAUSAL_LM


In [24]:
print(base_model.config.quantization_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [25]:
print("Loaded in 4-bit:", base_model.is_loaded_in_4bit)
print("Device map:", getattr(base_model, "hf_device_map", None))

Loaded in 4-bit: True
Device map: None


In [26]:
print("Memory footprint in GB:", model.get_memory_footprint() / 1024**3)

Memory footprint in GB: 0.7122690677642822


In [27]:
for name, param in model.named_parameters():
    print(name, param.dtype, param.device, param.requires_grad)
    break

base_model.model.model.embed_tokens.weight torch.float16 cuda:0 False


In [29]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    report_to="none",
    remove_unused_columns=False,
)


In [30]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)

Adding EOS to train dataset:   0%|          | 0/2508 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2508 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2508 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2508 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [32]:
trainer.train()

RuntimeError: gemm input type at::BFloat16 and output type float is only supported for CUDA devices with compute capability 8.0 or higher

In [33]:
#saving lora adapter
model.save_pretrained(f"{OUTPUT_DIR}/adapter")

In [34]:
tokenizer.save_pretrained(f"{OUTPUT_DIR}/adapter")

('./TinyLlama/adapter/tokenizer_config.json',
 './TinyLlama/adapter/chat_template.jinja',
 './TinyLlama/adapter/tokenizer.json')

In [35]:
base_model=MODEL_NAME

In [36]:
base_model

'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

In [38]:
# Reload the base model correctly using Hugging Face AutoModel
from transformers import AutoModelForCausalLM
import torch

real_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

inference_model = PeftModel.from_pretrained(real_base_model, f"{OUTPUT_DIR}/adapter")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [39]:
inference_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [40]:
prompt = """<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
If you want to achieve greatness, stop asking for permission.<end_of_turn>
<start_of_turn>model
"""

In [41]:
inputs = tokenizer(prompt, return_tensors="pt").to(inference_model.device)

In [42]:
with torch.no_grad():
    output = inference_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [43]:
print(tokenizer.decode(output[0], skip_special_tokens=True))

<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
If you want to achieve greatness, stop asking for permission.<end_of_turn>
<start_of_turn>model
Generate relevant tags for the following text:

Text:
The most important lesson I learned from this experience is that you can’t stop learning. The only thing that truly matters is that you keep pushing forward, even when it feels
